In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
import tensorflow as tf
import random
import os
from IPython.display import display
from sklearn.metrics import accuracy_score

In [4]:
# --- 4. paramètres ---
TIME_STEPS = 20
FUTURE_STEPS = 8
nb_pred = 10
seuil = 0.5 # seuil de décision pour trade

In [5]:
# --- 1. Chargement des données ---
df = pd.read_csv("Données_Diff.csv", parse_dates=True, index_col=0)
df = df.sort_index()

# --- Nettoyage AVANT normalisation ---
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=df.columns)  # supprime toutes les lignes avec NaN/Inf

# --- 2. Préparation des features et target ---
features = df.drop(columns=["result"])
target = df["result"]
y_binary = np.where(target == 1, 1, 0)

# --- 3. Normalisation ---
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(features)

# --- 4. créer les séries ---
def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_binary, TIME_STEPS)
X_train, y_train = X_seq[:-FUTURE_STEPS], y_seq[:-FUTURE_STEPS]

# Pour prédire les x dernières données du CSV
X_test_last_n = X_seq[-FUTURE_STEPS:]
y_test_last_n = y_seq[-FUTURE_STEPS:]

# Sélection aléatoire dans les 20% les plus récentes
nb_seq = len(X_seq)
start_20p = int(nb_seq * 0.8)
end_20p = nb_seq - FUTURE_STEPS
random_starts = random.sample(range(start_20p, end_20p), nb_pred)
X_tests = [X_seq[start:start+FUTURE_STEPS] for start in random_starts]
y_tests = [y_seq[start:start+FUTURE_STEPS] for start in random_starts]

Entrainer un seul model 

In [218]:
model = Sequential()
model.add(Input(shape=(TIME_STEPS, X_seq.shape[2])))
model.add(GRU(64))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

# Entraînement du modèle
history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.2, verbose=0)

# Accuracy sur l'entraînement
probas_train = model.predict(X_test_last_n, verbose=0).flatten()
preds_train = (probas_train > 0.5).astype(int)
accuracy_train = accuracy_score(y_test_last_n, preds_train)
print(f"Accuracy sur l'entraînement : {accuracy_train:.2%}")

# Accuracy sur la validation (si tu veux la meilleure val_accuracy pendant l'entraînement)
val_accuracies = history.history['val_accuracy']
print(f"Meilleure accuracy sur la validation (pendant fit) : {max(val_accuracies):.2%}")

model.save("Model_GRU_64_2.keras")

Accuracy sur l'entraînement : 37.50%
Meilleure accuracy sur la validation (pendant fit) : 55.85%


Choisir le meilleur model

In [ ]:
best_val_acc = 0
best_model = None
best_seed = None

for seed in random.sample(range(1,1000), 5):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
    model = Sequential()
    model.add(Input(shape=(TIME_STEPS, X_seq.shape[2])))
    model.add(GRU(64))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
    history = model.fit(
        X_seq, y_seq,
        epochs=200,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    val_acc = max(history.history['val_accuracy'])
    print(f"Seed {seed} : meilleure val_accuracy = {val_acc:.2%}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model = model
        best_seed = seed
        model.save("Best_GRU_Model_seq.keras")

print(f"\nMeilleur modèle : seed={best_seed}, val_accuracy={best_val_acc:.2%}")

#train => 55.01 % 
#seq => 54.26 % 

Seed 847 : meilleure val_accuracy = 53.63%
Seed 27 : meilleure val_accuracy = 53.00%
Seed 453 : meilleure val_accuracy = 53.52%
Seed 663 : meilleure val_accuracy = 53.63%
Seed 972 : meilleure val_accuracy = 54.26%

Meilleur modèle : seed=972, val_accuracy=54.26%


Pour des données aléatoires 

In [ ]:
model_filename = "Best_GRU_Model.keras"
model = load_model(model_filename)
all_precisions = []
all_precisions_hausse = []
all_sommes = []
all_preds_days = []
all_reals_days = []

for i, (X_test, y_test) in enumerate(zip(X_tests, y_tests)):
    probas = model.predict(X_test, verbose=0).flatten()
    preds = (probas > 0.5).astype(int)
    accuracy = np.mean(preds == y_test)
    all_precisions.append(accuracy)

    # Ajoute les prédictions et les valeurs réelles pour chaque jour
    all_preds_days.append(list(preds))
    all_reals_days.append(list(y_test))

    mask_hausse = preds == 1
    if np.any(mask_hausse):
        accuracy_hausse_total = np.mean(preds[mask_hausse] == y_test[mask_hausse])
        accuracy_hausse = list(preds[mask_hausse] == y_test[mask_hausse])
    else:
        accuracy_hausse = 1
        accuracy_hausse_total = 1
    all_precisions_hausse.append(accuracy_hausse_total)

    # Correction ici : on utilise i pour accéder à random_starts
    idx = df.index[random_starts[i]+TIME_STEPS : random_starts[i]+TIME_STEPS+FUTURE_STEPS]
    diff_jour = df.loc[idx, "Diff_jour"].values
    somme = diff_jour[probas > seuil].sum()
    all_sommes.append(somme)

# Moyennes et sommes globales
mean_precision = np.mean(all_precisions)
PnL_somme = np.sum(all_sommes)
precisions_percent = pd.Series([f"{acc:.2%}" for acc in all_precisions])
mean_precisions_hausse = np.mean(all_precisions_hausse)
precisions_hausse_percent = pd.Series([f"{acc3:.2%}" for acc3 in all_precisions_hausse])

print("Modèle utilisé :", model_filename)
print("\nNombre de jours prédit    :", FUTURE_STEPS)
print("Nombre de prédictions     :",nb_pred)
print("Seuil pour le trade       :", seuil)
print(f"Précision sur les hausses : {mean_precisions_hausse:.2%}")
print(f"Précision des modèles     : {mean_precision:.2%}")
print(f"PnL sur la série          : {PnL_somme:.3f} points")

Modèle utilisé : Best_GRU_Model.keras

Nombre de jours prédit    : 3
Nombre de prédictions     : 10
Seuil pour le trade       : 0.5
Précision sur les hausses : 63.33%
Précision des modèles     : 50.00%
PnL sur la série          : 104.390 points


In [161]:
# Afficher les détails des prédictions
print(f"\nPrécision des modèles : ")
display(precisions_percent)
print(f"\nPrécision des hausses : ")
display(precisions_hausse_percent)

for i in range(len(all_preds_days)):
    print(f"\nTest {i+1} :")
    print("  Prédictions :", ["hausse" if p == 1 else "baisse" for p in all_preds_days[i]])
    print("  Réels       :", ["hausse" if r == 1 else "baisse" for r in all_reals_days[i]])


Précision des modèles : 


0    33.33%
1    66.67%
dtype: object


Précision des hausses : 


0    100.00%
1     50.00%
dtype: object


Test 1 :
  Prédictions : ['baisse', 'baisse', 'baisse']
  Réels       : ['hausse', 'hausse', 'baisse']

Test 2 :
  Prédictions : ['hausse', 'hausse', 'baisse']
  Réels       : ['baisse', 'hausse', 'baisse']


Prédiction en utilisant les derniers jours. 

In [ ]:
#model_filename = "Model_GRU_64_2.keras"
model_filename = "Best_GRU_Model_train.keras"
model = load_model(model_filename)
all_precisions = []
all_precisions_hausse = []
all_sommes = []
all_preds_days = []
all_reals_days = []


probas_last10 = model.predict(X_test_last_n, verbose=0).flatten()
preds_last10 = (probas_last10 > 0.5).astype(int)

# On récupère les index des x derniers jours dans le DataFrame d'origine
idx_last10 = df.index[-FUTURE_STEPS:]
diff_jour_last10 = df.loc[idx_last10, "Diff_jour"].values

# Résultat des jours prédit à la hausse
somme = diff_jour_last10[probas_last10 > seuil].sum() 
all_sommes.append(somme)

# Construction du tableau de comparaison
compare_df = pd.DataFrame({
    "Jour": [f"J-{FUTURE_STEPS-i}" for i in range(FUTURE_STEPS, 0, -1)],
    "P(hausse)": np.round(probas_last10, 3),
    "P(baisse)": np.round(1 - probas_last10, 3),
    "Prédiction": ["Hausse" if p else "Baisse" for p in preds_last10],
    "Réalité": ["Hausse" if y else "Baisse" for y in y_test_last_n]
})

# Calcul de la précision sur les 10 derniers jours
accuracy = np.mean(preds_last10 == y_test_last_n)

# Précision uniquement sur les cas où le modèle prédit une hausse
mask_hausse = preds_last10 == 1
if np.any(mask_hausse):
    accuracy_hausse = np.mean(preds_last10[mask_hausse] == y_test_last_n[mask_hausse])
else:
    accuracy_hausse = np.nan  # Aucun cas de hausse prédit

print("Modèle utilisé :", model_filename)
print("\nNombre de jours prédit    :", FUTURE_STEPS)
print("Seuil pour le trade       :", seuil)
print(f"Précision sur les hausses : {accuracy_hausse:.2%}")
print(f"Précision du modèles     : {accuracy:.2%}")
print(f"PnL sur la série          : {somme:.3f} points")

from tabulate import tabulate
print("\n📊 Comparaison sur les 10 derniers jours réels du CSV :")
print(tabulate(compare_df, headers='keys', tablefmt='github', showindex=False))

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "conv1d_2" is incompatible with the layer: expected axis -1 of input shape to have value 25, but received input with shape (8, 20, 74)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(8, 20, 74), dtype=float32)
  • training=False
  • mask=None